## About Dataset:

Name: Jigsaw Toxic Text Classification

Source: Kaggle

In [1]:
import pandas as pd
import numpy as np
import torch
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, multilabel_confusion_matrix

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm

In [3]:
#Load and Understand the dataset
train_df = pd.read_csv("train.csv")

print(train_df.shape)
print(train_df.head())

(159571, 8)
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


In [5]:
#Data Preprocessing

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["comment_text"] = train_df["comment_text"].astype(str).apply(clean_text)

# Drop missing
train_df = train_df.dropna(subset=["comment_text"])

In [37]:
#Label the Dataset

labels_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

train_df["labels"] = train_df[labels_cols].values.tolist()

In [9]:
#Reduce Sample Size

train_df = train_df.sample(8000, random_state=42)   # safe size

In [11]:
#Data Splitting

train_data, temp_data = train_test_split(train_df, test_size=0.2, random_state=42)
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(len(train_data), len(val_data), len(test_data))

6400 800 800


In [13]:
#Tokenization
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [15]:
class ToxicDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts.tolist()
        self.labels = labels.tolist()

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=128,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }

In [17]:
train_loader = DataLoader(ToxicDataset(train_data["comment_text"], train_data["labels"]),
                          batch_size=16, shuffle=True, num_workers=0)

val_loader = DataLoader(ToxicDataset(val_data["comment_text"], val_data["labels"]),
                        batch_size=16, num_workers=0)

test_loader = DataLoader(ToxicDataset(test_data["comment_text"], test_data["labels"]),
                         batch_size=16, num_workers=0)

In [19]:
#Model Building

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=6,
    problem_type="multi_label_classification"
)

model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [21]:
#Fine-Tuning

optimizer = AdamW(model.parameters(), lr=2e-5)

In [23]:
def train(model, loader):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [25]:
#Model Evaluation

def evaluate(model, loader):
    model.eval()
    preds, true = [], []

    with torch.no_grad():
        for batch in tqdm(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            preds.extend(torch.sigmoid(logits).cpu().numpy())
            true.extend(labels.cpu().numpy())

    return np.array(preds), np.array(true)

In [27]:
def compute_metrics(preds, labels):
    preds = (preds > 0.5).astype(int)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="micro"),
        "recall": recall_score(labels, preds, average="micro"),
        "f1": f1_score(labels, preds, average="micro")
    }

In [29]:
def show_confusion_matrix(labels, preds):
    preds = (preds > 0.5).astype(int)
    cm = multilabel_confusion_matrix(labels, preds)
    print(cm)

In [31]:
for epoch in range(2):   # keep small first
    print(f"\nEpoch {epoch+1}")

    loss = train(model, train_loader)

    preds, labels = evaluate(model, val_loader)
    metrics = compute_metrics(preds, labels)

    print("Loss:", loss)
    print(metrics)


Epoch 1


100%|██████████| 50/50 [00:10<00:00,  4.94it/s]


Loss: 0.12264660839922727
{'accuracy': 0.92875, 'precision': 0.8, 'recall': 0.5906040268456376, 'f1': 0.6795366795366795}

Epoch 2


100%|██████████| 50/50 [00:10<00:00,  4.83it/s]

Loss: 0.0479700940893963
{'accuracy': 0.91375, 'precision': 0.7196969696969697, 'recall': 0.6375838926174496, 'f1': 0.6761565836298933}


In [33]:
preds, labels = evaluate(model, test_loader)

metrics = compute_metrics(preds, labels)
print("\nFinal Test Metrics:", metrics)

show_confusion_matrix(labels, preds)

100%|██████████| 50/50 [00:10<00:00,  4.79it/s]


Final Test Metrics: {'accuracy': 0.9025, 'precision': 0.7486910994764397, 'recall': 0.7079207920792079, 'f1': 0.727735368956743}
[[[690  23]
  [ 12  75]]

 [[787   1]
  [ 12   0]]

 [[740  11]
  [ 13  36]]

 [[800   0]
  [  0   0]]

 [[740  12]
  [ 16  32]]

 [[793   1]
  [  6   0]]]
